In [13]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Navigate to the directory containing the .tar file
%cd /content/drive/MyDrive/RUGD Dataset/tarFolder/

# Unzip the .tar file
!tar -xvf rugd-DatasetNinja.tar

In [14]:
import os

root = "/content/drive/MyDrive/RUGD Dataset/tarFolder/train"

print(os.listdir(root))

['ann', 'img', 'meta']


In [15]:
os.listdir("/content/drive/MyDrive/RUGD Dataset/tarFolder/train/meta")

['park-2_00001.png.json',
 'park-2_00006.png.json',
 'park-2_00011.png.json',
 'park-2_00016.png.json',
 'park-2_00021.png.json',
 'park-2_00026.png.json',
 'park-2_00031.png.json',
 'park-2_00036.png.json',
 'park-2_00041.png.json',
 'park-2_00046.png.json',
 'park-2_00051.png.json',
 'park-2_00056.png.json',
 'park-2_00061.png.json',
 'park-2_00066.png.json',
 'park-2_00071.png.json',
 'park-2_00076.png.json',
 'park-2_00081.png.json',
 'park-2_00086.png.json',
 'park-2_00091.png.json',
 'park-2_00096.png.json',
 'park-2_00101.png.json',
 'park-2_00106.png.json',
 'park-2_00111.png.json',
 'park-2_00116.png.json',
 'park-2_00121.png.json',
 'park-2_00126.png.json',
 'park-2_00131.png.json',
 'park-2_00136.png.json',
 'park-2_00141.png.json',
 'park-2_00146.png.json',
 'park-2_00151.png.json',
 'park-2_00156.png.json',
 'park-2_00161.png.json',
 'park-2_00166.png.json',
 'park-2_00171.png.json',
 'park-2_00176.png.json',
 'park-2_00181.png.json',
 'park-2_00186.png.json',
 'park-2_001

In [34]:
import json
import os

meta_dir = "/content/drive/MyDrive/RUGD Dataset/tarFolder/train/ann"

file = os.listdir(meta_dir)[3]

with open(os.path.join(meta_dir, file)) as f:
    data = json.load(f)

print(data)

{'description': '', 'tags': [{'id': 16788340, 'tagId': 29910, 'name': 'park', 'value': 2, 'labelerLogin': 'inbox@datasetninja.com', 'createdAt': '2024-01-31T12:46:59.872Z', 'updatedAt': '2024-01-31T12:46:59.872Z'}], 'size': {'height': 550, 'width': 688}, 'objects': [{'id': 139293558, 'classId': 6512325, 'objectId': None, 'description': '', 'geometryType': 'bitmap', 'labelerLogin': 'inbox@datasetninja.com', 'createdAt': '2024-01-31T12:46:59.878Z', 'updatedAt': '2024-01-31T12:46:59.878Z', 'tags': [], 'classTitle': 'sky', 'bitmap': {'data': 'eJwBiAd3+IlQTkcNChoKAAAADUlIRFIAAAKwAAAAtwEDAAAA0BD/ZAAAAAZQTFRFAAAA////pdmf3QAAAAF0Uk5TAEDm2GYAAAcwSURBVHic1dvPat82HABweS5xYSEu7BJYiPMIv7HL7zDqFvYg3XroNbdlEGKXQHtbXmDPsirkkN7yCHPpYZeyOezi0d+sSbL1X/JPkv3r6BfSyInzqSLJkiwpAI3RFBCAZAOkQCyRo9cI9TRZDXdf4o/3XYpsQW8UbKOzvWBzhH7R2Pc3XmzeAC0k9qBECU02gwuqPq3PHGyBWoltdZYXQtaBsldZcu1gcQi2zUy2ZqkOZJuBHUsBFPhXOXex7yQWbHS2AGDN0kkns29AvnGy+D88ldhKZ3GcGV+hNxekXMCFR5W14LGppkbBkPtLzJLwY7+zsND4Uon68tUBTR47ylZmO3CoC42d3ZT

In [20]:
import base64
import zlib
import numpy as np
from PIL import Image
import io

bitmap_data = data["objects"][0]["bitmap"]["data"]

decoded = base64.b64decode(bitmap_data)

decompressed = zlib.decompress(decoded)

mask = Image.open(io.BytesIO(decompressed))

mask_np = np.array(mask)

print(mask_np.shape)

(183, 688)


In [23]:
import numpy as np

height = data["size"]["height"]
width = data["size"]["width"]

segmentation = np.zeros((height, width), dtype=np.uint8)

for i, obj in enumerate(data["objects"]):

    class_id = i + 1

    bitmap_data = obj["bitmap"]["data"]

    decoded = base64.b64decode(bitmap_data)
    decompressed = zlib.decompress(decoded)

    mask = Image.open(io.BytesIO(decompressed))
    mask = np.array(mask)

    y_origin, x_origin = obj["bitmap"]["origin"]

    mask_h, mask_w = mask.shape

    # Calculate the intersection dimensions to ensure the mask fits within segmentation
    y_start = y_origin
    y_end = min(y_origin + mask_h, height)
    x_start = x_origin
    x_end = min(x_origin + mask_w, width)

    # Crop the mask to the intersection dimensions
    cropped_mask = mask[0 : y_end - y_start, 0 : x_end - x_start]

    # Apply the cropped mask to the segmentation array
    if cropped_mask.size > 0: # Ensure the cropped mask is not empty
        segmentation[y_start:y_end, x_start:x_end][cropped_mask > 0] = class_id

In [26]:
from PIL import Image
import numpy as np
import os

# The 'root' and 'file' variables are available in the kernel state from previous cells.
# root = '/content/drive/MyDrive/RUGD Dataset/tarFolder/train'
# file = 'park-2_00016.png.json'

image_filename = file.replace('.json', '')
img_path = os.path.join(root, 'img', image_filename)

image = Image.open(img_path)
image = np.array(image)

import tensorflow as tf

dataset = tf.data.Dataset.from_tensor_slices((image, segmentation))

dataset = dataset.map(lambda x, y: (tf.cast(x, tf.float32)/255.0, y))

dataset = dataset.batch(8)

In [27]:
import os
import json
import base64
import zlib
import io

import numpy as np
from PIL import Image
import tensorflow as tf

In [3]:
import os

root = "/content/drive/MyDrive/RUGD Dataset/tarFolder/train"

ann_path = os.path.join(root, "ann")
img_path = os.path.join(root, "img")

In [29]:
images = []
masks = []

In [20]:
def create_rugd_generator(ann_path, img_path):
    def rugd_generator_impl():
        import os
        import json
        import base64
        import zlib
        import io

        import numpy as np
        from PIL import Image

        for file in os.listdir(ann_path):

            if not file.endswith(".json"):
                continue

            json_path = os.path.join(ann_path, file)

            with open(json_path) as f:
                data = json.load(f)

            height = data["size"]["height"]
            width = data["size"]["width"]

            segmentation = np.zeros((height, width), dtype=np.uint8)

            for i, obj in enumerate(data["objects"]):

                class_id = i + 1

                bitmap_data = obj["bitmap"]["data"]

                decoded = base64.b64decode(bitmap_data)
                decompressed = zlib.decompress(decoded)

                mask = Image.open(io.BytesIO(decompressed))
                mask = np.array(mask)

                y_origin, x_origin = obj["bitmap"]["origin"]

                mask_h, mask_w = mask.shape

                y_start = max(y_origin, 0)
                x_start = max(x_origin, 0)

                y_end = min(y_origin + mask_h, height)
                x_end = min(x_origin + mask_w, width)

                if y_start >= y_end or x_start >= x_end:
                    continue

                mask_y_start = y_start - y_origin
                mask_y_end = mask_y_start + (y_end - y_start)

                mask_x_start = x_start - x_origin
                mask_x_end = mask_x_start + (x_end - x_start)

                cropped_mask = mask[mask_y_start:mask_y_end, mask_x_start:mask_x_end]

                segmentation[y_start:y_end, x_start:x_end][cropped_mask > 0] = class_id

            image_filename = file.replace(".json", "")
            image_file = os.path.join(img_path, image_filename)

            image = Image.open(image_file)
            image = np.array(image)

            yield image, segmentation
    return rugd_generator_impl

In [21]:
import tensorflow as tf

generator_callable = create_rugd_generator(ann_path, img_path)

dataset = tf.data.Dataset.from_generator(
    generator_callable,
    output_signature=(
        tf.TensorSpec(shape=(550,688,3), dtype=tf.uint8),
        tf.TensorSpec(shape=(550,688), dtype=tf.uint8),
    )
)

In [22]:
dataset = dataset.map(
    lambda x,y: (tf.cast(x, tf.float32)/255.0, y)
)

dataset = dataset.batch(8).prefetch(tf.data.AUTOTUNE)

In [23]:
for img, mask in dataset.take(1):
    print(img.shape)
    print(mask.shape)

(8, 550, 688, 3)
(8, 550, 688)


To iterate through the entire `tf.data.Dataset`:

In [25]:
for image_batch, mask_batch in dataset:
    print(f"Image Batch Shape: {image_batch.shape}, Mask Batch Shape: {mask_batch.shape}")
    # Add your training or evaluation logic here

Image Batch Shape: (8, 550, 688, 3), Mask Batch Shape: (8, 550, 688)
Image Batch Shape: (8, 550, 688, 3), Mask Batch Shape: (8, 550, 688)
Image Batch Shape: (8, 550, 688, 3), Mask Batch Shape: (8, 550, 688)
Image Batch Shape: (8, 550, 688, 3), Mask Batch Shape: (8, 550, 688)
Image Batch Shape: (8, 550, 688, 3), Mask Batch Shape: (8, 550, 688)
Image Batch Shape: (8, 550, 688, 3), Mask Batch Shape: (8, 550, 688)
Image Batch Shape: (8, 550, 688, 3), Mask Batch Shape: (8, 550, 688)
Image Batch Shape: (8, 550, 688, 3), Mask Batch Shape: (8, 550, 688)
Image Batch Shape: (8, 550, 688, 3), Mask Batch Shape: (8, 550, 688)
Image Batch Shape: (8, 550, 688, 3), Mask Batch Shape: (8, 550, 688)
Image Batch Shape: (8, 550, 688, 3), Mask Batch Shape: (8, 550, 688)
Image Batch Shape: (8, 550, 688, 3), Mask Batch Shape: (8, 550, 688)
Image Batch Shape: (8, 550, 688, 3), Mask Batch Shape: (8, 550, 688)
Image Batch Shape: (8, 550, 688, 3), Mask Batch Shape: (8, 550, 688)
Image Batch Shape: (8, 550, 688, 3

In [28]:
for image_batch, mask_batch in dataset.take(1):
    print(np.unique(mask_batch))

[ 0  1  2  3  4  5  6  7  8  9 10]


In [29]:
for image_batch, mask_batch in dataset.take(1):
    num_classes = len(np.unique(mask_batch))
    print("Number of classes:", num_classes)

Number of classes: 11


In [30]:
mask = tf.expand_dims(mask, -1)

In [35]:
import numpy as np
from PIL import Image
import os

# This line is added to define 'image' for demonstration purposes.
# In a typical workflow with tf.data.Dataset, images are processed within the dataset pipeline.
# The 'root' and 'file' variables are assumed to be defined from previous cells.
image_filename = file.replace('.json', '') # 'file' is from cell zApCI1H605A2, e.g., 'park-2_00016.png.json'
img_path = os.path.join(root, 'img', image_filename) # 'root' is from cell e9QgzdbM3kq8

image = Image.open(img_path)
image = np.array(image) # Convert to numpy array to match prior context

image = image / 255.0

In [14]:
import os
import json

ann_path = "/content/drive/MyDrive/RUGD Dataset/tarFolder/train/ann"

classes = set()

for file in os.listdir(ann_path):
    if not file.endswith(".json"):
        continue

    with open(os.path.join(ann_path, file)) as f:
        data = json.load(f)

    for obj in data["objects"]:
        classes.add(obj["classTitle"])

print("Number of classes:", len(classes))
print("Classes:", classes)

Number of classes: 23
Classes: {'vehicle', 'tree', 'gravel', 'fence', 'picnic-table', 'pole', 'sand', 'grass', 'container', 'mulch', 'building', 'asphalt', 'log', 'dirt', 'water', 'sign', 'sky', 'person', 'rock-bed', 'concrete', 'bush', 'bicycle', 'rock'}


In [ ]:
import os
import json
import base64
import zlib
import io
import numpy as np
from PIL import Image
import tensorflow as tf

# Paths
ann_path = "/content/drive/MyDrive/RUGD Dataset/tarFolder/train/ann"
img_path = "/content/drive/MyDrive/RUGD Dataset/tarFolder/train/img"

images = []
labels = []

# Classes considered obstacles
obstacle_classes = [
    "rock", "rock-bed", "tree", "bush", "log",
    "person", "vehicle", "bicycle",
    "building", "fence", "pole", "sign",
    "container", "picnic-table", "water"
]

for file in os.listdir(ann_path):

    if not file.endswith(".json"):
        continue

    json_file = os.path.join(ann_path, file)

    with open(json_file) as f:
        data = json.load(f)

    height = data["size"]["height"]
    width = data["size"]["width"]

    total_pixels = height * width
    obstacle_pixels = 0

    # Compute obstacle coverage
    for obj in data["objects"]:

        class_name = obj["classTitle"]

        if class_name not in obstacle_classes:
            continue

        bitmap_data = obj["bitmap"]["data"]

        decoded = base64.b64decode(bitmap_data)
        decompressed = zlib.decompress(decoded)

        mask = Image.open(io.BytesIO(decompressed))
        mask = np.array(mask)

        obstacle_pixels += np.sum(mask > 0)

    obstacle_percent = obstacle_pixels / total_pixels

    # Traversibility rule
    if obstacle_percent >= 0.30:
        label = 1   # NOT traversible
    else:
        label = 0   # traversible

    # Load corresponding image
    image_filename = file.replace(".json", "")
    image_file = os.path.join(img_path, image_filename)

    # Safety check
    if not os.path.exists(image_file):
        print("Missing image:", image_file)
        continue

    image = Image.open(image_file)
    image = np.array(image)

    images.append(image)
    labels.append(label)

# Convert to numpy
images = np.array(images)
labels = np.array(labels)

print("Images shape:", images.shape)
print("Labels shape:", labels.shape)

# Create TensorFlow dataset
dataset = tf.data.Dataset.from_tensor_slices((images, labels))

dataset = dataset.map(lambda x, y: (tf.cast(x, tf.float32) / 255.0, y))

dataset = dataset.shuffle(1000).batch(8)

print("Dataset ready")

Images shape: (4779, 550, 688, 3)
Labels shape: (4779,)
